# 01 数据概览与字段理解

本Notebook主要用于对Citi Bike骑行数据进行初步理解和质量检查。分析目标包括：读取原始压缩包数据，确认数据粒度、字段结构、时间范围、缺失情况、用户类型分布、车型分布以及骑行时长异常情况。

本项目第一版先使用2024年7月单月数据跑通完整分析流程，后续可根据需要扩展到多个代表月份或全年数据。

## 1 导入依赖并设置项目路径

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import zipfile

DATA_RAW_DIR = Path("../data_raw")
DATA_CLEAN_DIR = Path("../data_clean")
OUTPUT_DIR = Path("../outputs")

DATA_CLEAN_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

zip_path = DATA_RAW_DIR / "202407-citibike-tripdata.zip"

## 2 查看压缩包内部文件结构

Citi Bike 的月度数据通常以 zip 压缩包形式提供。部分月份的压缩包中可能包含多个 CSV 文件，因此在正式读取前，先查看压缩包内部文件列表，确认需要读取哪些数据文件。

In [4]:
with zipfile.ZipFile(zip_path, "r") as z:
    file_list = z.namelist()

file_list[:10]

['202407-citibike-tripdata_2.csv',
 '202407-citibike-tripdata_3.csv',
 '202407-citibike-tripdata_1.csv',
 '202407-citibike-tripdata_4.csv',
 '202407-citibike-tripdata_5.csv']

**观察与总结**

该月度压缩包中包含 5 个 CSV 文件，说明 2024 年 7 月数据量较大，官方将单月数据拆分为多个文件存放。因此后续读取时不能只读取其中一个 CSV，而需要遍历压缩包内所有 CSV 并合并，否则会低估当月骑行量。

## 3 读取并合并月度骑行数据

In [6]:
dfs = []

dtype_map = {
    "ride_id": "string",
    "rideable_type": "string",
    "start_station_name": "string",
    "start_station_id": "string",
    "end_station_name": "string",
    "end_station_id": "string",
    "member_casual": "string",
}

with zipfile.ZipFile(zip_path, "r") as z:
    csv_files = [name for name in z.namelist() if name.endswith(".csv")]
    
    for csv_file in csv_files:
        with z.open(csv_file) as f:
            temp = pd.read_csv(
                f,
                dtype=dtype_map,
                low_memory=False,
            )
            dfs.append(temp)

trip_raw = pd.concat(dfs, ignore_index=True)

trip_raw.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,984F632114B98410,electric_bike,2024-07-11 13:32:52.359,2024-07-11 13:41:04.825,9 Ave & W 18 St,6190.08,9 Ave & W 33 St,6492.08,40.743174,-74.003664,40.752568,-73.996765,casual
1,9B0E546FDB460C0E,electric_bike,2024-07-13 13:18:42.179,2024-07-13 13:22:46.631,W 42 St & 6 Ave,6517.08,W 49 St & 8 Ave,6747.06,40.754920,-73.984550,40.762272,-73.987882,member
2,6B939445A283D985,classic_bike,2024-07-08 20:34:27.848,2024-07-08 20:41:46.350,8 Ave & W 52 St,6816.07,9 Ave & W 33 St,6492.08,40.763707,-73.985162,40.752568,-73.996765,member
3,49444E058931E427,electric_bike,2024-07-14 15:42:44.695,2024-07-14 15:55:54.771,W 120 St & Claremont Ave,7745.07,W 78 St & Broadway,7311.07,40.810949,-73.963400,40.783400,-73.980931,casual
4,74033CB639411DA0,classic_bike,2024-07-09 08:23:38.797,2024-07-09 08:28:48.647,W 42 St & 6 Ave,6517.08,W 49 St & 8 Ave,6747.06,40.754920,-73.984550,40.762272,-73.987882,member


## 4 查看数据规模和基础结构

In [7]:
print("行数:", len(trip_raw))
print("列数:", trip_raw.shape[1])

display(trip_raw.head())
display(trip_raw.info())

行数: 4722896
列数: 13


,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,984F632114B98410,electric_bike,2024-07-11 13:32:52.359,2024-07-11 13:41:04.825,9 Ave & W 18 St,6190.08,9 Ave & W 33 St,6492.08,40.743174,-74.003664,40.752568,-73.996765,casual
1,9B0E546FDB460C0E,electric_bike,2024-07-13 13:18:42.179,2024-07-13 13:22:46.631,W 42 St & 6 Ave,6517.08,W 49 St & 8 Ave,6747.06,40.754920,-73.984550,40.762272,-73.987882,member
2,6B939445A283D985,classic_bike,2024-07-08 20:34:27.848,2024-07-08 20:41:46.350,8 Ave & W 52 St,6816.07,9 Ave & W 33 St,6492.08,40.763707,-73.985162,40.752568,-73.996765,member
3,49444E058931E427,electric_bike,2024-07-14 15:42:44.695,2024-07-14 15:55:54.771,W 120 St & Claremont Ave,7745.07,W 78 St & Broadway,7311.07,40.810949,-73.963400,40.783400,-73.980931,casual
4,74033CB639411DA0,classic_bike,2024-07-09 08:23:38.797,2024-07-09 08:28:48.647,W 42 St & 6 Ave,6517.08,W 49 St & 8 Ave,6747.06,40.754920,-73.984550,40.762272,-73.987882,member


<class 'pandas.DataFrame'>
RangeIndex: 4722896 entries, 0 to 4722895
Data columns (total 13 columns):
 #   Column              Dtype  
---  ------              -----  
 0   ride_id             string 
 1   rideable_type       string 
 2   started_at          str    
 3   ended_at            str    
 4   start_station_name  string 
 5   start_station_id    string 
 6   end_station_name    string 
 7   end_station_id      string 
 8   start_lat           float64
 9   start_lng           float64
 10  end_lat             float64
 11  end_lng             float64
 12  member_casual       string 
dtypes: float64(4), str(2), string(7)
memory usage: 468.4 MB


None

**观察与总结**

合并后的原始数据共有 4,722,896 行、13 列，说明 2024 年 7 月单月骑行量已经达到百万级，数据规模足以支撑时间、站点和用户类型维度的运营分析。

字段类型方面，`ride_id`、站点名称、站点ID、车型和用户类型已按字符串读取；起终点经纬度为数值型。`started_at` 和 `ended_at` 此时仍为字符串类型，后续需要转换为 datetime 类型，才能计算骑行时长并提取日期、小时、星期等时间维度。

## 5 查看字段名称

In [8]:
trip_raw.columns.tolist()

['ride_id',
 'rideable_type',
 'started_at',
 'ended_at',
 'start_station_name',
 'start_station_id',
 'end_station_name',
 'end_station_id',
 'start_lat',
 'start_lng',
 'end_lat',
 'end_lng',
 'member_casual']

**观察与总结**

从字段结构看，数据主要包含五类信息：骑行唯一标识、车型信息、起止时间、起终点站点与经纬度、用户类型。

其中，`ride_id` 可用于判断骑行记录唯一性；`started_at` 和 `ended_at` 是后续时间趋势与骑行时长分析的基础；`start_station_id`、`end_station_id` 以及对应经纬度是后续站点需求和流入流出失衡分析的基础；`member_casual` 可用于区分会员用户和临时用户行为差异。

## 6 检查缺失值情况

In [9]:
missing_summary = (
    trip_raw
    .isna()
    .sum()
    .to_frame("missing_count")
)

missing_summary["missing_rate"] = (
    missing_summary["missing_count"] / len(trip_raw)
).round(4)

display(missing_summary)

,missing_count,missing_rate
ride_id,0,0.0000
rideable_type,0,0.0000
started_at,0,0.0000
ended_at,0,0.0000
start_station_name,2640,0.0006
start_station_id,2640,0.0006
end_station_name,12567,0.0027
end_station_id,13575,0.0029
start_lat,2640,0.0006
start_lng,2640,0.0006


**观察与总结**

整体缺失情况较轻，`ride_id`、`rideable_type`、`started_at`、`ended_at` 和 `member_casual` 均无缺失，说明核心骑行事件、时间和用户类型字段较完整。

缺失主要集中在站点和经纬度字段：起点站名称、起点站ID、起点经纬度缺失率约为 0.06%；终点站名称、终点站ID、终点经纬度缺失率约为 0.27% 到 0.29%。这类缺失比例较低，但会影响站点流入流出、站点地图和站点失衡分析。

因此，后续构建 `trip_base` 时可以保留全量记录用于整体骑行量和用户类型分析；但在站点失衡和地图分析中，需要使用起终点站点字段完整的样本。

## 7 转换时间字段

原始数据中的开始时间和结束时间需要转换为 datetime 类型，才能计算骑行时长，并提取日期、小时和星期等时间维度。这些字段是后续分析骑行需求高峰、工作日与周末差异的基础。

In [13]:
trip_raw["started_at"] = pd.to_datetime(trip_raw["started_at"], errors="coerce")
trip_raw["ended_at"] = pd.to_datetime(trip_raw["ended_at"], errors="coerce")

trip_raw["ride_duration_min"] = (
    trip_raw["ended_at"] - trip_raw["started_at"]
).dt.total_seconds() / 60

trip_raw["start_date"] = trip_raw["started_at"].dt.date
trip_raw["start_hour"] = trip_raw["started_at"].dt.hour
trip_raw["day_of_week"] = trip_raw["started_at"].dt.day_name()

display(
    trip_raw.head()
)

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,ride_duration_min,start_date,start_hour,day_of_week
0,984F632114B98410,electric_bike,2024-07-11 13:32:52.359,2024-07-11 13:41:04.825,9 Ave & W 18 St,6190.08,9 Ave & W 33 St,6492.08,40.743174,-74.003664,40.752568,-73.996765,casual,8.207767,2024-07-11,13,Thursday
1,9B0E546FDB460C0E,electric_bike,2024-07-13 13:18:42.179,2024-07-13 13:22:46.631,W 42 St & 6 Ave,6517.08,W 49 St & 8 Ave,6747.06,40.754920,-73.984550,40.762272,-73.987882,member,4.074200,2024-07-13,13,Saturday
2,6B939445A283D985,classic_bike,2024-07-08 20:34:27.848,2024-07-08 20:41:46.350,8 Ave & W 52 St,6816.07,9 Ave & W 33 St,6492.08,40.763707,-73.985162,40.752568,-73.996765,member,7.308367,2024-07-08,20,Monday
3,49444E058931E427,electric_bike,2024-07-14 15:42:44.695,2024-07-14 15:55:54.771,W 120 St & Claremont Ave,7745.07,W 78 St & Broadway,7311.07,40.810949,-73.963400,40.783400,-73.980931,casual,13.167933,2024-07-14,15,Sunday
4,74033CB639411DA0,classic_bike,2024-07-09 08:23:38.797,2024-07-09 08:28:48.647,W 42 St & 6 Ave,6517.08,W 49 St & 8 Ave,6747.06,40.754920,-73.984550,40.762272,-73.987882,member,5.164167,2024-07-09,8,Tuesday


## 8 汇总数据基本概况

本段从整体层面汇总骑行记录数、唯一骑行ID数、时间范围、站点数量、车型数量、用户类型数量和骑行时长分布。该表可以帮助我们快速确认数据覆盖范围和后续分析对象。

In [14]:
overview = pd.Series(
    {
        "records": len(trip_raw),
        "unique_rides": trip_raw["ride_id"].nunique(),
        "start_time_min": trip_raw["started_at"].min(),
        "start_time_max": trip_raw["started_at"].max(),
        "start_stations": trip_raw["start_station_id"].nunique(),
        "end_stations": trip_raw["end_station_id"].nunique(),
        "rideable_types": trip_raw["rideable_type"].nunique(),
        "member_types": trip_raw["member_casual"].nunique(),
        "avg_duration_min": trip_raw["ride_duration_min"].mean(),
        "median_duration_min": trip_raw["ride_duration_min"].median(),
    }
).to_frame("value")

display(overview)

,value
records,4722896
unique_rides,4722896
start_time_min,2024-06-30 00:52:38.502000
start_time_max,2024-07-31 23:57:41.895000
start_stations,2239
end_stations,2252
rideable_types,2
member_types,2
avg_duration_min,14.549678
median_duration_min,9.65785


**观察与总结**

本月数据共有 4722896 条骑行记录，`unique_rides` 与总行数一致，初步说明 `ride_id` 可以作为骑行事件唯一标识。

时间范围为 2024-06-30 00:52:38 至 2024-07-31 23:57:41。虽然文件名为 202407，但数据中出现了少量 6 月 30 日的边界记录。后续如果要严格分析 2024 年 7 月自然月，应在 `02_build_trip_base.ipynb` 中明确是否过滤为 `2024-07-01 <= started_at < 2024-08-01`。

站点方面，起点站数量为2239，终点站数量为2252，说明数据覆盖站点较多，适合做站点需求和流入流出失衡分析。

骑行时长方面，平均骑行时长约 14.55 分钟，中位数约 9.66 分钟。平均值高于中位数，说明骑行时长分布可能右偏，存在少量较长骑行记录，需要在后续异常值检查中进一步处理。

## 9 检查用户类型分布

Citi Bike数据中的member_casual字段用于区分会员用户和临时用户。会员用户通常更接近通勤场景，临时用户可能更接近游客或偶发出行场景，因此该字段是后续用户行为差异分析的重要维度。

In [15]:
member_summary = (
    trip_raw["member_casual"]
    .value_counts()
    .rename_axis("member_casual")
    .reset_index(name="rides")
)

member_summary["ride_share"] = (
    member_summary["rides"] / member_summary["rides"].sum()
).round(4)

display(member_summary)

,member_casual,rides,ride_share
0,member,3632220,0.7691
1,casual,1090676,0.2309


**观察与总结**

从用户类型看，会员用户贡献 3632220 次骑行，占比 76.91%；临时用户贡献 1090676 次骑行，占比 23.09%。会员用户是 Citi Bike 的主要使用群体。

这说明后续分析不能只看整体骑行量，还需要区分会员和临时用户。会员用户可能更接近日常通勤需求，临时用户可能更接近游客或偶发出行需求，两类用户在骑行时段、骑行距离、车型选择和站点分布上可能存在明显差异。

## 10 检查车型分布

本段统计不同车型的骑行次数和占比。车型分布可以帮助我们理解用户对普通自行车和电动自行车的使用偏好，也为后续分析不同用户类型的车型选择差异做准备。

In [16]:
bike_type_summary = (
    trip_raw["rideable_type"]
    .value_counts()
    .rename_axis("rideable_type")
    .reset_index(name="rides")
)

bike_type_summary["ride_share"] = (
    bike_type_summary["rides"] / bike_type_summary["rides"].sum()
).round(4)

display(bike_type_summary)

,rideable_type,rides,ride_share
0,electric_bike,3114491,0.6594
1,classic_bike,1608405,0.3406


**观察与总结**

从车型分布看，电动自行车骑行次数为 3114491，占比 65.94%；普通自行车骑行次数为 1608405，占比 34.06%。电动自行车已经是本月更主要的使用车型。

后续可以进一步分析不同用户类型对车型的偏好。例如，临时用户是否更偏好电动自行车，会员用户是否在通勤时段更多使用某类车型。这有助于理解车辆投放结构和用户需求之间的关系。

## 11 检查骑行时长分布和异常值

骑行时长是共享单车分析中的重要指标。过短、为负数或过长的骑行记录可能来自数据异常、测试记录或特殊使用场景。后续构建清洗数据表时，需要根据本段结果制定异常值处理规则。

In [17]:
duration_summary = trip_raw["ride_duration_min"].describe(
    percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]
).to_frame("ride_duration_min")

display(duration_summary)

print("骑行时长 <= 0 的记录数:", (trip_raw["ride_duration_min"] <= 0).sum())
print("骑行时长 > 24小时 的记录数:", (trip_raw["ride_duration_min"] > 24 * 60).sum())

,ride_duration_min
count,4.722896e+06
mean,1.454968e+01
std,3.433712e+01
min,7.746667e-02
1%,1.408516e+00
5%,2.458833e+00
50%,9.657850e+00
95%,3.642138e+01
99%,7.056055e+01
max,1.500492e+03


骑行时长 <= 0 的记录数: 0
骑行时长 > 24小时 的记录数: 1294


**观察与总结**

骑行时长整体呈明显右偏分布。中位数约为 9.66 分钟，95% 的骑行不超过约 36.42 分钟，99% 的骑行不超过约 70.56 分钟，但最大值达到约 1500.49 分钟，说明存在少量极长骑行记录。

骑行时长小于等于 0 的记录数为 0，说明没有明显的负时长或零时长异常。但骑行时长超过 24 小时的记录有 1294 条，这类记录占比很低，但会拉高平均骑行时长，影响用户行为和车型使用分析。

因此，后续构建 `trip_base` 时建议保留一个清洗口径，例如剔除 `ride_duration_min <= 0` 和 `ride_duration_min > 24 * 60` 的记录。整体需求量统计可以说明是否基于清洗后样本，避免异常长骑行影响均值指标。

## 12 检查ride_id唯一性

本项目的基础粒度是一行一次骑行，因此需要检查ride_id是否唯一。如果ride_id唯一，可以将其视为骑行事件的唯一标识；如果存在重复，则需要进一步判断是否为重复记录或数据异常。

In [19]:
ride_id_check = pd.Series(
    {
        "records": len(trip_raw),
        "unique_ride_id": trip_raw["ride_id"].nunique(),
        "duplicated_ride_id": len(trip_raw) - trip_raw["ride_id"].nunique(),
    }
).to_frame("value")

display(ride_id_check)

print("每行记录代表一次 Citi Bike 骑行事件。")
print("ride_id 是否唯一:", trip_raw["ride_id"].is_unique)

,value
records,4722896
unique_ride_id,4722896
duplicated_ride_id,0


每行记录代表一次 Citi Bike 骑行事件。
ride_id 是否唯一: True


**观察与总结**

`ride_id` 唯一值数量与总记录数一致，重复 `ride_id` 数量为 0，说明本数据中每一行可以视为一次唯一骑行事件。

因此，本项目的基础分析粒度可以定义为“一行一次骑行”。后续在按日期、小时、站点或用户类型聚合时，骑行次数可以直接使用记录数或 `ride_id` 去重数进行统计。

## 总结

通过本 Notebook 的初步检查，可以确认 2024 年 7 月 Citi Bike 原始数据已成功读取并合并。该压缩包包含 5 个 CSV 文件，合并后共有 4722896 条骑行记录、13 个字段。

本数据的基础粒度为“一行一次骑行事件”。`ride_id` 唯一值数量与总行数一致，重复 `ride_id` 数量为 0，因此后续可以将 `ride_id` 作为骑行事件唯一标识，并基于事件级数据进一步聚合到日期、小时、站点和用户类型等维度。

字段结构方面，数据包含骑行ID、车型、开始时间、结束时间、起终点站点、起终点经纬度和用户类型。核心时间字段、车型字段和用户类型字段无缺失；站点和经纬度字段存在少量缺失，其中终点相关字段缺失率约为 0.27% 到 0.29%。因此，后续可以保留全量记录用于整体需求和用户类型分析，但在站点失衡和地图分析中需要使用起终点站点字段完整的样本。

时间范围方面，数据覆盖 2024-06-30 至 2024-07-31。由于文件名为 202407，但数据中包含少量 6 月 30 日边界记录，后续构建清洗表时需要明确是否过滤为严格的 2024 年 7 月自然月。

用户结构方面，会员用户贡献 3632220 次骑行，占比 76.91%；临时用户贡献 1090676 次骑行，占比 23.09%。说明会员是主要使用群体，后续有必要进一步分析会员与临时用户在骑行时段、车型偏好和站点分布上的差异。

车型结构方面，电动自行车贡献 3114491 次骑行，占比 65.94%；普通自行车贡献 1608405 次骑行，占比 34.06%。说明电动自行车是本月更主要的使用车型，后续可以进一步分析不同用户类型对车型的偏好差异。

骑行时长方面，中位数约为 9.66 分钟，平均值约为 14.55 分钟，说明大多数骑行属于短途出行。但最大骑行时长达到约 1500.49 分钟，且存在 1294 条超过 24 小时的记录。后续在 `02_build_trip_base.ipynb` 中需要制定异常时长清洗规则，避免极端值影响平均骑行时长等指标。

下一步将在 `02_build_trip_base.ipynb` 中构建清洗后的 `trip_base` 数据表，重点处理时间字段、骑行时长异常、站点缺失、自然月过滤，并新增日期、小时、星期、工作日/周末、骑行时段等分析字段，为后续需求规律、站点失衡和用户类型差异分析做准备。